In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

# Global Configuration Parameters
VOCAB_SIZE = 50257
CONTEXT_LENGTH = 256  # Set precisely to your updated 256 token checkpoint configuration
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 8
DROPOUT = 0.0

class GPTEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.position_embedding = nn.Embedding(CONTEXT_LENGTH, D_MODEL)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x, start_pos=0):
        B, T = x.shape
        positions = torch.arange(start_pos, start_pos + T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(positions)
        return self.dropout(tok + pos)

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert D_MODEL % N_HEADS == 0
        self.qkv = nn.Linear(D_MODEL, 3 * D_MODEL)
        self.proj = nn.Linear(D_MODEL, D_MODEL)
        self.resid_dropout = nn.Dropout(DROPOUT)

        # Mechanistic Interpretability Core Probes
        self.last_attention_weights = None
        self.last_q = None
        self.last_k = None
        self.last_v = None

    def forward(self, x, past_key_value=None):
        B, T, C = x.shape
        head_dim = C // N_HEADS

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        # Reshape to (Batch, Heads, Sequence, Head_Dimension)
        q = q.view(B, T, N_HEADS, head_dim).transpose(1, 2)
        k = k.view(B, T, N_HEADS, head_dim).transpose(1, 2)
        v = v.view(B, T, N_HEADS, head_dim).transpose(1, 2)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        current_kv = (k, v)

        # INTERPRETABILITY CAPTURE STEP: Save historical tensor maps to memory bounds
        self.last_q = q.detach()
        self.last_k = k.detach()
        self.last_v = v.detach()

        is_causal = True if (T > 1 and past_key_value is None) else False

        if self.training or T > 1:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
            if is_causal:
                mask = torch.tril(torch.ones(T, T, device=x.device))
                att = att.masked_fill(mask == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            self.last_attention_weights = att.detach()
            out = att @ v
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out)), current_kv

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D_MODEL, 4 * D_MODEL),
            nn.GELU(),
            nn.Linear(4 * D_MODEL, D_MODEL),
            nn.Dropout(DROPOUT)
        )
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.attn = MultiHeadAttention()
        self.ffn = FeedForward()

    def forward(self, x, past_key_value=None):
        attn_out, current_kv = self.attn(self.ln1(x), past_key_value)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, current_kv

class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = GPTEmbedding()
        self.blocks = nn.ModuleList([TransformerBlock() for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)

        self.lm_head.weight = self.embedding.token_embedding.weight
        self.last_hidden_states = []

    def forward(self, idx, start_pos=0, kv_cache=None):
        x = self.embedding(idx, start_pos=start_pos)

        new_cache = []
        self.last_hidden_states = []

        if kv_cache is None:
            kv_cache = [None for _ in range(N_LAYERS)]

        for block, past_kv in zip(self.blocks, kv_cache):
            x, current_kv = block(x, past_key_value=past_kv)
            new_cache.append(current_kv)
            self.last_hidden_states.append(x.detach())

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, new_cache

    @torch.inference_mode()
    def generate(self, prompt, max_new_tokens=100, temperature=0.7, top_k=40, device="cuda"):
        self.eval()
        enc = tiktoken.get_encoding("gpt2")
        ids = enc.encode(prompt)

        prompt_len = len(ids)
        max_allowed_generation = CONTEXT_LENGTH - prompt_len

        # Enforce explicit constraint configurations
        if max_allowed_generation <= 0:
            print(f"[!] Warning: Prompt length ({prompt_len}) meets or exceeds max sequence limits ({CONTEXT_LENGTH}). Generation aborted.")
            return prompt

        generation_steps = min(max_new_tokens, max_allowed_generation)
        if generation_steps < max_new_tokens:
            print(f"[*] Info: Generation budget adjusted down from {max_new_tokens} to {generation_steps} to honor absolute token barriers.")

        # Optimize storage arrays: maintain local copies directly
        all_ids = list(ids)
        idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

        # Process structural prefill state
        logits, cache = self(idx, start_pos=0, kv_cache=None)
        current_pos = prompt_len

        next_token_input = idx[:, -1:]

        generation_trace = []
        for _ in range(generation_steps):
            logits_step = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits_step, min(top_k, logits_step.size(-1)))
                logits_step[logits_step < v[:, -1, None]] = -float("inf")

            probs = F.softmax(logits_step, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Clean programmatic append operations
            all_ids.append(next_token.item())

            # Fast causal step
            logits, cache = self(next_token, start_pos=current_pos, kv_cache=cache)
            current_pos += 1

            generation_trace.append({

                "position": current_pos,
                "predicted_token_id":
                    next_token.item(),
                "context_ids":
                    list(all_ids),
                "final_logits":
                    logits[:, -1, :].detach().cpu(),
                "layer_hidden_states":[
                    h[:, -1, :].detach().cpu()
                    for h in self.last_hidden_states
                ]
            })
        return {
          "text": enc.decode(all_ids),
          "all_ids": all_ids,
          "trace": generation_trace
        }


        # Execute exactly one string text decode operation at the absolute end
        return enc.decode(all_ids)

In [ ]:
def load_production_inference_engine(checkpoint_path, device="cuda"):
    """
    Loads raw state dictionaries directly without structural translation friction.
    """
    print(f"Instantiating clean inference graph on target device: [{device}]")
    model = GPTModel()

    print(f"Loading checkpoint state: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Extract structural state dictionary matching keys natively
    state_dict = checkpoint if "model" not in checkpoint else checkpoint["model"]

    # Native direct instantiation
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print("Graph execution validated. Ready for inference and mechanistic probing.")
    return model

In [ ]:
model= load_production_inference_engine("/content/drive/MyDrive/GPT_from_scratch/checkpoints/checkpoint_31625.pt","cpu")

In [ ]:
model.generate("whats python",100,0.8,40,"cpu")